# Construção do Painel do Censo Escolar (2007–2019) — Notebook de Auditoria

**Disciplina:** Big Data & Analytics — trabalho final (exercício metodológico).

## 1. Objetivo deste notebook

Apresentar, de forma didática e reprodutível, a etapa que construiu o
**painel de presença federal de Educação Profissional e Tecnológica (EPT)**
dos municípios da Fase II da expansão da Rede Federal, a partir dos
microdados do Censo Escolar da Educação Básica.

Este notebook **não reconstrói** os dados. Ele:

- carrega os três Parquets **já produzidos e validados** pela rotina de produção
  [`src/constroi_painel_censo_escolar.py`](../src/constroi_painel_censo_escolar.py);
- reexecuta apenas **verificações leves**, tabelas e estatísticas descritivas;
- serve como trilha de auditoria para conferência direta.

> **Escopo causal:** nenhuma atribuição de tratamento, coorte ou efeito
> causal é feita aqui. As contagens de "presença federal EPT" descrevem
> apenas o que foi **observado** no Censo Escolar. Ver
> [`docs/methodology/CONTRATO_CAUSAL.md`](../docs/methodology/CONTRATO_CAUSAL.md).

## 2. Contexto do projeto

- **Pergunta de pesquisa:** efeito da chegada de novos campi da Rede Federal
  sobre a atividade econômica municipal.
- **Unidade de análise:** município-ano. **Janela:** 2007–2019.
- **Papel desta etapa:** medir, ano a ano, a presença de escolas federais e
  a oferta de EPT em cada município da Fase II, para servir de insumo às
  decisões metodológicas ainda abertas no contrato causal.
- **Documentos relacionados:**
  - [`docs/institutional/EXPANSAO_FASE_II.md`](../docs/institutional/EXPANSAO_FASE_II.md) — reconstrução institucional da política;
  - [`docs/institutional/AUDITORIA_LISTA_FASE_II.md`](../docs/institutional/AUDITORIA_LISTA_FASE_II.md) — lista das 150 cidades-polo → 147 municípios IBGE;
  - [`docs/data/AUDITORIA_CONSTRUCAO_CENSO_ESCOLAR.md`](../docs/data/AUDITORIA_CONSTRUCAO_CENSO_ESCOLAR.md) — auditoria detalhada desta construção;
  - [`docs/methodology/CONTRATO_CAUSAL.md`](../docs/methodology/CONTRATO_CAUSAL.md) — decisões causais ex ante.

## 3. Fontes utilizadas

Todas locais — **nenhum download é feito por este notebook**.

| Insumo | Caminho | Conteúdo |
|---|---|---|
| Microdados do Censo Escolar 2007–2019 | `data/raw/inep/censo_escolar/microdados_censo_escolar_{ano}.zip` | 13 arquivos ZIP do INEP; 1 CSV de escola por ano |
| Manifesto de origem | `data/raw/inep/censo_escolar/source_manifest.json` | URLs de origem e SHA-256 de cada ZIP |
| Lista de municípios da Fase II | `data/processed/fase_ii_municipios.parquet` | 147 municípios com código IBGE |

A célula abaixo apenas **confere a existência** dos insumos e do
manifesto (sem abrir os ZIPs).

In [1]:
import hashlib, io, json, sys, zipfile
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

# Resolve a raiz do projeto procurando a rotina de produção
ROOT = Path.cwd()
while not (ROOT / "src" / "constroi_painel_censo_escolar.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "src" / "constroi_painel_censo_escolar.py").exists(), "raiz do projeto não encontrada"

DATA_RAW       = ROOT / "data" / "raw" / "inep" / "censo_escolar"
DATA_INTERIM   = ROOT / "data" / "interim"
DATA_PROCESSED = ROOT / "data" / "processed"

P_ESCOLA  = DATA_INTERIM   / "censo_escolas_federais_2007_2019.parquet"
P_MUNANO  = DATA_INTERIM   / "censo_federal_municipio_ano_2007_2019.parquet"
P_PAINEL  = DATA_PROCESSED / "painel_presenca_federal_ept_fase_ii_2007_2019.parquet"
P_FASE_II = DATA_PROCESSED / "fase_ii_municipios.parquet"

print("Raiz do projeto:", ROOT)
print()

zips = sorted(DATA_RAW.glob("microdados_censo_escolar_*.zip"))
print(f"ZIPs do Censo Escolar encontrados: {len(zips)}")
for z in zips:
    print(f"  {z.name:38s} {z.stat().st_size/1e6:7.1f} MB")

manifesto = DATA_RAW / "source_manifest.json"
print("\nManifesto de origem:", "OK" if manifesto.exists() else "AUSENTE", f"({manifesto.name})")
_m = json.loads(manifesto.read_bytes().decode("utf-8-sig"))
print("  fonte :", _m["fonte"])
print("  período:", _m["periodo"])
print("  anos no manifesto:", [a["ano"] for a in _m["arquivos"]])

for p in (P_ESCOLA, P_MUNANO, P_PAINEL, P_FASE_II):
    print(("OK  " if p.exists() else "FALTA ") + str(p.relative_to(ROOT)))

Raiz do projeto: C:\GitHub\data-science-projects\projetos\big-data-expansao-rede-federal-economia-municipal

ZIPs do Censo Escolar encontrados: 13
  microdados_censo_escolar_2007.zip         21.3 MB
  microdados_censo_escolar_2008.zip         21.9 MB
  microdados_censo_escolar_2009.zip         23.7 MB
  microdados_censo_escolar_2010.zip         22.9 MB
  microdados_censo_escolar_2011.zip         25.9 MB
  microdados_censo_escolar_2012.zip         26.8 MB
  microdados_censo_escolar_2013.zip         27.8 MB
  microdados_censo_escolar_2014.zip         25.8 MB
  microdados_censo_escolar_2015.zip         26.7 MB
  microdados_censo_escolar_2016.zip         26.5 MB
  microdados_censo_escolar_2017.zip         25.7 MB
  microdados_censo_escolar_2018.zip         26.0 MB
  microdados_censo_escolar_2019.zip         26.2 MB

Manifesto de origem: OK (source_manifest.json)
  fonte : INEP - Microdados do Censo Escolar da Educação Básica
  período: 2007-2019
  anos no manifesto: [2007, 2008, 2009, 2010

## 4. Municípios da Fase II

A população geográfica desta etapa são os **147 municípios** (código IBGE)
correspondentes às 150 cidades-polo do Anexo I da Chamada Pública
MEC/SETEC nº 001/2007 — as 4 unidades do Distrito Federal colapsam em 1
código (Brasília), reduzindo 150 → 147. A reconstrução dessa lista e a
divergência com o "144" de documentos anteriores estão registradas em
[`AUDITORIA_LISTA_FASE_II.md`](../docs/institutional/AUDITORIA_LISTA_FASE_II.md).

In [2]:
df_fase_ii = pd.read_parquet(P_FASE_II)
fase_ii_mun = sorted(df_fase_ii["codigo_municipio_ibge"].astype(str).str.zfill(7).tolist())

print("Linhas em fase_ii_municipios.parquet :", len(df_fase_ii))
print("Códigos IBGE distintos               :", df_fase_ii["codigo_municipio_ibge"].nunique())
print("Códigos IBGE nulos                   :", df_fase_ii["codigo_municipio_ibge"].isna().sum())
print("Colunas                              :", list(df_fase_ii.columns))

display(df_fase_ii.head(8))

print("\nMunicípios da Fase II por UF:")
display(df_fase_ii.groupby("uf").size().rename("n_municipios").to_frame().T)

Linhas em fase_ii_municipios.parquet : 147
Códigos IBGE distintos               : 147
Códigos IBGE nulos                   : 0
Colunas                              : ['codigo_municipio_ibge', 'municipio', 'uf', 'quantidade_unidades_fase_ii', 'nomes_unidades', 'fonte_primaria', 'paginas_fonte', 'status_validacao', 'observacao']


,codigo_municipio_ibge,municipio,uf,quantidade_unidades_fase_ii,nomes_unidades,fonte_primaria,paginas_fonte,status_validacao,observacao
0,1200203,Cruzeiro do Sul,AC,1,CRUZEIRO DO SUL,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,
1,1200500,Sena Madureira,AC,1,SENA MADUREIRA,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,
2,2700300,Arapiraca,AL,1,ARAPIRACA,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,
3,2704500,Maragogi,AL,1,MARAGOGI,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,
4,2706703,Penedo,AL,1,PENEDO,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,
5,2707107,Piranhas,AL,1,PIRANHAS (XINGÓ),"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,anotacao entre parenteses na fonte: 'XINGÓ' (n...
6,1600279,Laranjal do Jari,AP,1,LARANJAL DO JARI,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,
7,1302405,Lábrea,AM,1,LÁBREA,"Chamada Publica MEC/SETEC no 001/2007, Anexo I",9,exata,



Municípios da Fase II por UF:


uf,AC,AL,AM,AP,BA,CE,DF,ES,GO,MA,MG,MS,MT,PA,PB,PE,PI,PR,RJ,RN,RO,RR,RS,SC,SE,SP,TO
n_municipios,2,4,5,1,8,6,1,5,6,8,12,5,6,5,5,5,6,6,7,6,2,1,10,7,3,12,3


## 5. Período de 2007 a 2019

A janela do projeto é **2007–2019, inclusive — 13 anos**. Todos os ZIPs
do Censo cobrem exatamente esse intervalo e as três tabelas usam o mesmo
conjunto de anos (conferido nas seções 11 e 13).

In [3]:
ANOS = list(range(2007, 2020))
ANOS_STR = [str(a) for a in ANOS]
print("Anos da janela:", ANOS_STR)
print("Total de anos :", len(ANOS))

Anos da janela: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']
Total de anos : 13


## 6. Explicação do pipeline de construção

A rotina de produção [`src/constroi_painel_censo_escolar.py`](../src/constroi_painel_censo_escolar.py)
executa, em ordem:

1. **Carrega a lista da Fase II** (`fase_ii_municipios.parquet`) → 147 códigos IBGE.
2. **Lê cada ZIP do Censo diretamente da memória** (sem extrair o CSV para
   disco): `zipfile` → bytes → `io.BytesIO` → `pandas.read_csv` em blocos
   de 50 000 linhas, `sep=";"`, `encoding="latin-1"` (ver seção 15),
   `usecols` com 19 colunas.
3. **Filtra escolas federais** (`TP_DEPENDENCIA == 1`) e deriva as flags
   (seção 7).
4. **Concatena** os 13 anos → tabela **escola-ano**; ordena por
   `(NU_ANO_CENSO, CO_ENTIDADE)`; valida e grava.
5. **Agrega para município-ano** (`groupby(NU_ANO_CENSO, CO_MUNICIPIO)`) →
   contagens de escolas e somas de matrículas/turmas/docentes EPT; valida
   e grava.
6. **Monta o painel Fase II**: grade completa `147 municípios × 13 anos`
   (`MultiIndex.from_product`) + `left join` com a município-ano; anos sem
   escola ficam com contagens 0 e flags `False`; identificação municipal
   completada pelo cadastro da Fase II; valida e grava.
7. **Audita entradas/saídas/lacunas** de presença EPT federal ativa —
   **apenas no log de console**, sem gravar nada nos Parquets e **sem
   decidir tratamento**.

> Este notebook **não repete** essa lógica. A célula abaixo apenas
> **transcreve** o trecho da rotina que define as flags, para conferência.

In [4]:
src_txt = (ROOT / "src" / "constroi_painel_censo_escolar.py").read_text(encoding="utf-8").splitlines()
# localiza o bloco "# Cria flags derivadas"
ini = next(i for i, l in enumerate(src_txt) if "Cria flags derivadas" in l)
print(f"--- src/constroi_painel_censo_escolar.py (linhas {ini+1}-{ini+10}) ---")
for i in range(ini, ini + 10):
    print(f"{i+1:4d} | {src_txt[i]}")

--- src/constroi_painel_censo_escolar.py (linhas 165-174) ---
 165 |     # Cria flags derivadas
 166 |     df["fl_em_atividade"] = (df["TP_SITUACAO_FUNCIONAMENTO"] == 1)
 167 |     df["fl_oferta_ept"] = (df["IN_PROF"] == 1)
 168 |     df["fl_oferta_ept_tecnica"] = (df["IN_PROF_TEC"] == 1)
 169 |     df["fl_presenca_federal_ept_ativa"] = (
 170 |         (df["TP_DEPENDENCIA"] == 1)
 171 |         & (df["TP_SITUACAO_FUNCIONAMENTO"] == 1)
 172 |         & (df["IN_PROF"] == 1)
 173 |     )
 174 | 


## 7. Definições das flags e das contagens

### Nível escola-ano (uma linha por escola federal por ano)

| Flag | Definição | Observação |
|---|---|---|
| `fl_em_atividade` | `TP_SITUACAO_FUNCIONAMENTO == 1` | escola em funcionamento no ano |
| `fl_oferta_ept` | `IN_PROF == 1` | oferta de educação profissional — **independe** da situação de funcionamento |
| `fl_oferta_ept_tecnica` | `IN_PROF_TEC == 1` | oferta de curso técnico |
| `fl_presenca_federal_ept_ativa` | `TP_DEPENDENCIA == 1` **e** `TP_SITUACAO_FUNCIONAMENTO == 1` **e** `IN_PROF == 1` | campus federal com EPT efetivamente em atividade |

### Nível município-ano e painel (agregações)

| Coluna | Como é obtida |
|---|---|
| `qt_escolas_federais` | contagem de escolas federais no município-ano |
| `qt_escolas_em_atividade` | soma de `fl_em_atividade` |
| `qt_escolas_com_ept` | soma de `fl_oferta_ept` |
| `qt_escolas_com_ept_tecnica` | soma de `fl_oferta_ept_tecnica` |
| `qt_escolas_federal_ept_ativa` | soma de `fl_presenca_federal_ept_ativa` |
| `qt_mat_prof`, `qt_mat_prof_tec` | soma de matrículas em EPT / EPT técnica |
| `qt_tur_prof`, `qt_tur_prof_tec` | soma de turmas |
| `qt_doc_prof`, `qt_doc_prof_tec` | soma de docentes |
| `fl_presenca_federal` (painel) | `qt_escolas_federais > 0` |
| `fl_presenca_federal_ept` (painel) | `qt_escolas_com_ept > 0` |
| `fl_presenca_federal_ept_ativa` (painel) | `qt_escolas_federal_ept_ativa > 0` |

### Relações conceituais corretas entre os conjuntos

- `ept_ativa ⊆ ept`
- `ept_ativa ⊆ em_atividade`
- `ept ⊆ escolas_federais`
- `em_atividade ⊆ escolas_federais`

> **Não** se exige `ept ⊆ em_atividade`: uma escola pode ter registro de
> oferta de EPT num ano em que não está em atividade. Essa relação **não é
> apresentada nem testada** neste notebook.

## 8. Leitura dos três Parquets

Resultados principais da etapa, carregados **como estão**:

| Tabela | Arquivo | Grão |
|---|---|---|
| escola-ano | `data/interim/censo_escolas_federais_2007_2019.parquet` | escola federal × ano |
| município-ano | `data/interim/censo_federal_municipio_ano_2007_2019.parquet` | município com ≥1 escola federal × ano |
| painel Fase II | `data/processed/painel_presenca_federal_ept_fase_ii_2007_2019.parquet` | 147 municípios × 13 anos (grade completa) |

In [5]:
df_escola = pd.read_parquet(P_ESCOLA)
df_munano = pd.read_parquet(P_MUNANO)
df_painel = pd.read_parquet(P_PAINEL)

print("escola-ano   :", df_escola.shape)
print("município-ano:", df_munano.shape)
print("painel Fase II:", df_painel.shape)

escola-ano   : (6781, 23)
município-ano: (5192, 16)
painel Fase II: (1911, 19)


## 9. Dimensões, colunas e tipos

In [6]:
def resumo_schema(df, nome):
    r = pd.DataFrame({"coluna": df.columns, "dtype": [str(t) for t in df.dtypes],
                      "nao_nulos": df.notna().sum().values, "nulos": df.isna().sum().values})
    print(f"### {nome} — {df.shape[0]:,} linhas × {df.shape[1]} colunas")
    display(r)

resumo_schema(df_escola, "escola-ano")
resumo_schema(df_munano, "município-ano")
resumo_schema(df_painel, "painel Fase II")

### escola-ano — 6,781 linhas × 23 colunas


,coluna,dtype,nao_nulos,nulos
0,NU_ANO_CENSO,str,6781,0
1,SG_UF,str,6781,0
2,CO_UF,str,6781,0
3,NO_MUNICIPIO,str,6781,0
4,CO_MUNICIPIO,str,6781,0
5,CO_ENTIDADE,str,6781,0
6,NO_ENTIDADE,str,6781,0
7,TP_DEPENDENCIA,int64,6781,0
8,TP_LOCALIZACAO,int64,6781,0
9,TP_LOCALIZACAO_DIFERENCIADA,int64,6781,0


### município-ano — 5,192 linhas × 16 colunas


,coluna,dtype,nao_nulos,nulos
0,NU_ANO_CENSO,str,5192,0
1,CO_MUNICIPIO,str,5192,0
2,NO_MUNICIPIO,str,5192,0
3,SG_UF,str,5192,0
4,CO_UF,str,5192,0
5,qt_escolas_federais,Int64,5192,0
6,qt_escolas_em_atividade,Int64,5192,0
7,qt_escolas_com_ept,Int64,5192,0
8,qt_escolas_com_ept_tecnica,Int64,5192,0
9,qt_escolas_federal_ept_ativa,Int64,5192,0


### painel Fase II — 1,911 linhas × 19 colunas


,coluna,dtype,nao_nulos,nulos
0,CO_MUNICIPIO,str,1911,0
1,NU_ANO_CENSO,str,1911,0
2,NO_MUNICIPIO,str,1911,0
3,SG_UF,str,1911,0
4,CO_UF,str,1911,0
5,qt_escolas_federais,Int64,1911,0
6,qt_escolas_em_atividade,Int64,1911,0
7,qt_escolas_com_ept,Int64,1911,0
8,qt_escolas_com_ept_tecnica,Int64,1911,0
9,qt_escolas_federal_ept_ativa,Int64,1911,0


## 10. Amostras das tabelas

Primeiras linhas de cada tabela (ordenação determinística já aplicada na
produção).

In [7]:
display(df_escola.head(5))
display(df_munano.head(5))
display(df_painel.head(5))

,NU_ANO_CENSO,SG_UF,CO_UF,NO_MUNICIPIO,CO_MUNICIPIO,CO_ENTIDADE,NO_ENTIDADE,TP_DEPENDENCIA,TP_LOCALIZACAO,TP_LOCALIZACAO_DIFERENCIADA,TP_SITUACAO_FUNCIONAMENTO,IN_PROF,IN_PROF_TEC,QT_MAT_PROF,QT_MAT_PROF_TEC,QT_DOC_PROF,QT_DOC_PROF_TEC,QT_TUR_PROF,QT_TUR_PROF_TEC,fl_em_atividade,fl_oferta_ept,fl_oferta_ept_tecnica,fl_presenca_federal_ept_ativa
0,2007,RO,11,Ariquemes,1100023,11007877,ESCOLA AGROPECUARIA R CEPLAC RO EMARC,1,2,0,1,1.0,1.0,224.0,224.0,19.0,19.0,9.0,9.0,True,True,True,True
1,2007,RO,11,Colorado do Oeste,1100064,11037016,ESCOLA AGROTECNICA FEDERAL DE COLORADO DO OESTE,1,2,0,1,1.0,1.0,353.0,353.0,31.0,31.0,12.0,12.0,True,True,True,True
2,2007,AC,12,Rio Branco,1200401,12011410,ESC COLEGIO DE APLICAÇAO,1,1,0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,True,False,False,False
3,2007,AM,13,Manaus,1302603,13026844,ESC AGROTECNICA FEDERAL DE MANAUS,1,1,0,1,1.0,1.0,480.0,480.0,44.0,44.0,14.0,14.0,True,True,True,True
4,2007,AM,13,Manaus,1302603,13029916,CENTRO FEDERAL DE EDUC. TECNOLÓGICA DO AMAZONAS,1,1,0,1,1.0,1.0,1389.0,1389.0,63.0,63.0,48.0,48.0,True,True,True,True


,NU_ANO_CENSO,CO_MUNICIPIO,NO_MUNICIPIO,SG_UF,CO_UF,qt_escolas_federais,qt_escolas_em_atividade,qt_escolas_com_ept,qt_escolas_com_ept_tecnica,qt_escolas_federal_ept_ativa,qt_mat_prof,qt_mat_prof_tec,qt_tur_prof,qt_tur_prof_tec,qt_doc_prof,qt_doc_prof_tec
0,2007,1100023,Ariquemes,RO,11,1,1,1,1,1,224,224,9,9,19,19
1,2007,1100064,Colorado do Oeste,RO,11,1,1,1,1,1,353,353,12,12,31,31
2,2007,1200401,Rio Branco,AC,12,1,1,0,0,0,0,0,0,0,0,0
3,2007,1301209,Coari,AM,13,1,1,1,1,1,233,233,6,6,19,19
4,2007,1302603,Manaus,AM,13,3,3,3,3,3,2589,2589,82,82,166,166


,CO_MUNICIPIO,NU_ANO_CENSO,NO_MUNICIPIO,SG_UF,CO_UF,qt_escolas_federais,qt_escolas_em_atividade,qt_escolas_com_ept,qt_escolas_com_ept_tecnica,qt_escolas_federal_ept_ativa,qt_mat_prof,qt_mat_prof_tec,qt_tur_prof,qt_tur_prof_tec,qt_doc_prof,qt_doc_prof_tec,fl_presenca_federal,fl_presenca_federal_ept,fl_presenca_federal_ept_ativa
0,1100122,2007,Ji-Paraná,RO,11,0,0,0,0,0,0,0,0,0,0,0,False,False,False
1,1100122,2008,Ji-Paraná,RO,11,0,0,0,0,0,0,0,0,0,0,0,False,False,False
2,1100122,2009,Ji-Paraná,RO,11,1,1,1,1,1,248,248,7,7,23,23,True,True,True
3,1100122,2010,Ji-Paraná,RO,11,1,1,1,1,1,485,485,14,14,32,32,True,True,True
4,1100122,2011,Ji-Paraná,RO,11,1,1,1,1,1,616,616,18,18,31,31,True,True,True


## 11. Cobertura anual

Quantas escolas federais entram na tabela escola-ano em cada ano, e
confirmação de que o painel tem os 147 municípios em **todos** os 13 anos.

In [8]:
cob_escola = (df_escola.groupby("NU_ANO_CENSO")
              .agg(escolas_federais=("CO_ENTIDADE", "size"),
                   em_atividade=("fl_em_atividade", "sum"),
                   com_ept=("fl_oferta_ept", "sum"),
                   com_ept_tecnica=("fl_oferta_ept_tecnica", "sum"),
                   ept_federal_ativa=("fl_presenca_federal_ept_ativa", "sum")))
display(cob_escola)

print("Anos na escola-ano   :", sorted(df_escola["NU_ANO_CENSO"].unique().tolist()))
print("Anos no município-ano:", sorted(df_munano["NU_ANO_CENSO"].unique().tolist()))
print("Anos no painel       :", sorted(df_painel["NU_ANO_CENSO"].unique().tolist()))

mun_por_ano = df_painel.groupby("NU_ANO_CENSO")["CO_MUNICIPIO"].nunique()
print("\nMunicípios distintos por ano no painel: min =", mun_por_ano.min(),
      "| max =", mun_por_ano.max(), "(esperado 147 em todos)")

,escolas_federais,em_atividade,com_ept,com_ept_tecnica,ept_federal_ativa
NU_ANO_CENSO,,,,,
2007,235,235,180,180,180
2008,271,271,207,207,207
2009,302,302,239,239,239
2010,354,354,282,282,282
2011,470,451,388,388,388
2012,518,490,430,429,430
2013,523,512,453,453,453
2014,561,543,485,485,485
2015,647,637,574,574,574


Anos na escola-ano   : ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']
Anos no município-ano: ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']
Anos no painel       : ['2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019']

Municípios distintos por ano no painel: min = 147 | max = 147 (esperado 147 em todos)


## 12. Validação das chaves e duplicidades

| Tabela | Chave | Esperado |
|---|---|---|
| escola-ano | `(NU_ANO_CENSO, CO_ENTIDADE)` | 0 duplicatas, 0 nulos |
| município-ano | `(NU_ANO_CENSO, CO_MUNICIPIO)` | 0 duplicatas, 0 nulos |
| painel Fase II | `(CO_MUNICIPIO, NU_ANO_CENSO)` | 0 duplicatas, 0 nulos |

In [9]:
def checa_chave(df, chave, nome):
    dup = int(df.duplicated(subset=chave).sum())
    nul = int(df[chave].isna().any(axis=1).sum())
    print(f"{nome:16s} chave={chave}")
    print(f"    duplicatas: {dup}   linhas com algum nulo na chave: {nul}")
    return dup == 0 and nul == 0

ok12 = []
ok12.append(checa_chave(df_escola, ["NU_ANO_CENSO", "CO_ENTIDADE"], "escola-ano"))
ok12.append(checa_chave(df_munano, ["NU_ANO_CENSO", "CO_MUNICIPIO"], "município-ano"))
ok12.append(checa_chave(df_painel, ["CO_MUNICIPIO", "NU_ANO_CENSO"], "painel"))

print("\nComprimento dos códigos (escola-ano):")
print("  CO_ENTIDADE == 8 :", int((df_escola["CO_ENTIDADE"].str.len() == 8).all()))
print("  CO_MUNICIPIO == 7:", int((df_escola["CO_MUNICIPIO"].str.len() == 7).all()))
print("  CO_UF == 2       :", int((df_escola["CO_UF"].str.len() == 2).all()))

print("\nTodas as chaves OK:", all(ok12))

escola-ano       chave=['NU_ANO_CENSO', 'CO_ENTIDADE']


    duplicatas: 0   linhas com algum nulo na chave: 0
município-ano    chave=['NU_ANO_CENSO', 'CO_MUNICIPIO']
    duplicatas: 0   linhas com algum nulo na chave: 0
painel           chave=['CO_MUNICIPIO', 'NU_ANO_CENSO']
    duplicatas: 0   linhas com algum nulo na chave: 0

Comprimento dos códigos (escola-ano):
  CO_ENTIDADE == 8 : 1
  CO_MUNICIPIO == 7: 1
  CO_UF == 2       : 1

Todas as chaves OK: True


## 13. Validação dos 147 municípios e das 1.911 observações

O painel deve ser a **grade completa** `147 × 13 = 1.911`, com o conjunto
de municípios **idêntico** ao de `fase_ii_municipios.parquet`.

In [10]:
n_mun  = df_painel["CO_MUNICIPIO"].nunique()
n_anos = df_painel["NU_ANO_CENSO"].nunique()
n_obs  = len(df_painel)

print(f"Municípios distintos no painel : {n_mun}   (esperado 147)")
print(f"Anos distintos no painel       : {n_anos}   (esperado 13)")
print(f"Linhas no painel               : {n_obs}   (esperado 1.911 = 147 × 13)")

set_painel = set(df_painel["CO_MUNICIPIO"].unique())
set_fase_ii = set(fase_ii_mun)
print(f"\nConjunto painel == conjunto Fase II : {set_painel == set_fase_ii}")
print(f"  municípios da Fase II ausentes do painel : {len(set_fase_ii - set_painel)}")
print(f"  municípios no painel fora da Fase II     : {len(set_painel - set_fase_ii)}")

grade = df_painel.groupby("CO_MUNICIPIO")["NU_ANO_CENSO"].nunique()
print(f"\nTodo município tem exatamente 13 anos : {bool((grade == 13).all())}")

assert (n_mun, n_anos, n_obs) == (147, 13, 1911)
assert set_painel == set_fase_ii
print("\n>>> 147 municípios e 1.911 observações CONFIRMADOS.")

Municípios distintos no painel : 147   (esperado 147)
Anos distintos no painel       : 13   (esperado 13)
Linhas no painel               : 1911   (esperado 1.911 = 147 × 13)

Conjunto painel == conjunto Fase II : True
  municípios da Fase II ausentes do painel : 0
  municípios no painel fora da Fase II     : 0

Todo município tem exatamente 13 anos : True

>>> 147 municípios e 1.911 observações CONFIRMADOS.


## 14. Coerência das flags e das contagens

### 14.1 Relações de contenção (as quatro corretas)

- `ept_ativa ⊆ ept`
- `ept_ativa ⊆ em_atividade`
- `ept ⊆ escolas_federais`
- `em_atividade ⊆ escolas_federais`

### 14.2 Coerência entre flags e contagens no painel

- `fl_presenca_federal` ⇔ `qt_escolas_federais > 0`
- `fl_presenca_federal_ept` ⇔ `qt_escolas_com_ept > 0`
- `fl_presenca_federal_ept_ativa` ⇔ `qt_escolas_federal_ept_ativa > 0`
- implicações: `ept_ativa ⇒ ept ⇒ presença_federal`

In [11]:
res = {}

# --- 14.1 no nível escola-ano (conjuntos atômicos) ---
e = df_escola
res["esc: ept_ativa ⊆ ept"]          = bool((~e["fl_presenca_federal_ept_ativa"] | e["fl_oferta_ept"]).all())
res["esc: ept_ativa ⊆ em_atividade"] = bool((~e["fl_presenca_federal_ept_ativa"] | e["fl_em_atividade"]).all())
res["esc: toda escola é federal (TP_DEPENDENCIA==1)"] = bool((e["TP_DEPENDENCIA"] == 1).all())

# --- 14.1 no nível de contagens (município-ano e painel) ---
for nome, d in [("mun", df_munano), ("painel", df_painel)]:
    res[f"{nome}: ept_ativa ⊆ ept"]          = bool((d["qt_escolas_federal_ept_ativa"] <= d["qt_escolas_com_ept"]).all())
    res[f"{nome}: ept_ativa ⊆ em_atividade"] = bool((d["qt_escolas_federal_ept_ativa"] <= d["qt_escolas_em_atividade"]).all())
    res[f"{nome}: ept ⊆ escolas_federais"]   = bool((d["qt_escolas_com_ept"]          <= d["qt_escolas_federais"]).all())
    res[f"{nome}: em_atividade ⊆ escolas_federais"] = bool((d["qt_escolas_em_atividade"] <= d["qt_escolas_federais"]).all())

# --- 14.2 coerência flag <-> contagem no painel ---
p = df_painel
res["painel: fl_presenca_federal ⇔ qt>0"]           = bool((p["fl_presenca_federal"]           == (p["qt_escolas_federais"] > 0)).all())
res["painel: fl_presenca_federal_ept ⇔ qt>0"]       = bool((p["fl_presenca_federal_ept"]       == (p["qt_escolas_com_ept"] > 0)).all())
res["painel: fl_presenca_federal_ept_ativa ⇔ qt>0"] = bool((p["fl_presenca_federal_ept_ativa"] == (p["qt_escolas_federal_ept_ativa"] > 0)).all())
res["painel: ept_ativa ⇒ ept"]      = bool((~p["fl_presenca_federal_ept_ativa"] | p["fl_presenca_federal_ept"]).all())
res["painel: ept ⇒ presença_federal"] = bool((~p["fl_presenca_federal_ept"] | p["fl_presenca_federal"]).all())

# --- contagens não negativas ---
qt_cols = [c for c in df_painel.columns if c.startswith("qt_")]
res["painel: nenhuma contagem negativa"] = bool((df_painel[qt_cols] >= 0).all().all())
res["mun: nenhuma contagem negativa"]    = bool((df_munano[[c for c in df_munano.columns if c.startswith('qt_')]] >= 0).all().all())

# --- linhas do painel sem escola: flags devem ser todas False ---
sem = df_painel[df_painel["qt_escolas_federais"] == 0]
res["painel: linhas sem escola têm todas as flags False"] = bool(
    (~sem[["fl_presenca_federal", "fl_presenca_federal_ept", "fl_presenca_federal_ept_ativa"]].any(axis=1)).all())

tab = pd.DataFrame({"verificação": list(res), "passou": list(res.values())})
display(tab)
print("TODAS as verificações da seção 14 passaram:", all(res.values()))

,verificação,passou
0,esc: ept_ativa ⊆ ept,True
1,esc: ept_ativa ⊆ em_atividade,True
2,esc: toda escola é federal (TP_DEPENDENCIA==1),True
3,mun: ept_ativa ⊆ ept,True
4,mun: ept_ativa ⊆ em_atividade,True
5,mun: ept ⊆ escolas_federais,True
6,mun: em_atividade ⊆ escolas_federais,True
7,painel: ept_ativa ⊆ ept,True
8,painel: ept_ativa ⊆ em_atividade,True
9,painel: ept ⊆ escolas_federais,True


TODAS as verificações da seção 14 passaram: True


## 15. Verificação de nomes e ausência de *mojibake*

Os CSVs do Censo Escolar do INEP são **ISO-8859-1 (Latin-1)** e **não têm
BOM**. A primeira versão da rotina os lia como `utf-8-sig`, o que
corrompia todos os acentos com o caractere de substituição `�` (U+FFFD).
A correção (`encoding="latin-1"`) está descrita em
[`AUDITORIA_CONSTRUCAO_CENSO_ESCOLAR.md`](../docs/data/AUDITORIA_CONSTRUCAO_CENSO_ESCOLAR.md),
seção 10.

A célula abaixo confirma **0 ocorrências de `�`** nas colunas de nome das
três tabelas e mostra amostras acentuadas.

In [12]:
FFFD = "\ufffd"
def conta_mojibake(df, cols):
    return {c: int(df[c].astype(str).str.contains(FFFD, regex=False).sum()) for c in cols if c in df.columns}

print("escola-ano   :", conta_mojibake(df_escola, ["NO_MUNICIPIO", "NO_ENTIDADE", "SG_UF"]))
print("município-ano:", conta_mojibake(df_munano, ["NO_MUNICIPIO", "SG_UF"]))
print("painel       :", conta_mojibake(df_painel, ["NO_MUNICIPIO", "SG_UF"]))

acent = df_painel[df_painel["NO_MUNICIPIO"].str.contains("[áàâãéêíóôõúç]", case=False, regex=True)]
print("\nAmostra de municípios da Fase II com acento (painel):")
print(sorted(acent["NO_MUNICIPIO"].unique())[:15])

print("\nAmostra de nomes de escola com acento (escola-ano):")
ent = df_escola[df_escola["NO_ENTIDADE"].str.contains("[ÁÂÃÉÊÍÓÔÕÚÇ]", regex=True)]
print(ent["NO_ENTIDADE"].drop_duplicates().head(6).tolist())

escola-ano   : {'NO_MUNICIPIO': 0, 'NO_ENTIDADE': 0, 'SG_UF': 0}
município-ano: {'NO_MUNICIPIO': 0, 'SG_UF': 0}
painel       : {'NO_MUNICIPIO': 0, 'SG_UF': 0}

Amostra de municípios da Fase II com acento (painel):
['Acaraú', 'Alcântara', 'Angical do Piauí', 'Anápolis', 'Araguaína', 'Araçuaí', 'Avaré', 'Bagé', 'Barra do Garças', 'Bragança', 'Brasília', 'Caicó', 'Camaquã', 'Canindé', 'Conceição do Araguaia']

Amostra de nomes de escola com acento (escola-ano):
['ESC COLEGIO DE APLICAÇAO', 'CENTRO FEDERAL DE EDUC. TECNOLÓGICA DO AMAZONAS', 'EAF DE SÃO GABRIEL DA CACHOEIRA', 'CENTRO FEDERAL DE EDUCAÇÃO TECNOLÓGICA DE RORAIMA', 'CEDUC EDUCAÇÃO BÁSICA /UFRR', 'ESCOLA DE APLICAÇÃO DA UFPA']


## 16. Estatísticas descritivas básicas

> As contagens abaixo descrevem **presença observada no Censo Escolar**.
> Não constituem atribuição de tratamento, definição de coorte nem
> qualquer inferência causal.

In [13]:
# 16.1 — Rede federal e EPT, por ano (a partir da tabela escola-ano)
display(cob_escola.rename(columns={
    "escolas_federais": "escolas federais", "em_atividade": "em atividade",
    "com_ept": "com EPT", "com_ept_tecnica": "com EPT técnica",
    "ept_federal_ativa": "EPT federal ativa"}))

# 16.2 — Volumes de EPT somados por ano (município-ano)
vol = (df_munano.groupby("NU_ANO_CENSO")[["qt_mat_prof", "qt_tur_prof", "qt_doc_prof"]]
       .sum().rename(columns={"qt_mat_prof": "matrículas EPT",
                              "qt_tur_prof": "turmas EPT", "qt_doc_prof": "docentes EPT"}))
display(vol)

,escolas federais,em atividade,com EPT,com EPT técnica,EPT federal ativa
NU_ANO_CENSO,,,,,
2007,235,235,180,180,180
2008,271,271,207,207,207
2009,302,302,239,239,239
2010,354,354,282,282,282
2011,470,451,388,388,388
2012,518,490,430,429,430
2013,523,512,453,453,453
2014,561,543,485,485,485
2015,647,637,574,574,574


,matrículas EPT,turmas EPT,docentes EPT
NU_ANO_CENSO,,,
2007,114581,4455,8134
2008,132784,4889,10520
2009,158885,5581,12025
2010,179691,6370,14168
2011,205165,7492,17195
2012,225677,8472,19413
2013,241802,9241,21584
2014,249903,9764,23604
2015,325933,12123,27442


In [14]:
# 16.3 — Painel Fase II: nº de municípios com presença federal EPT ativa OBSERVADA por ano
pres_ano = (df_painel.groupby("NU_ANO_CENSO")
            .agg(municipios_com_presenca_federal=("fl_presenca_federal", "sum"),
                 municipios_com_ept=("fl_presenca_federal_ept", "sum"),
                 municipios_com_ept_ativa=("fl_presenca_federal_ept_ativa", "sum")))
display(pres_ano)

# 16.4 — dos 147, quantos têm presença EPT federal ativa observada em pelo menos um ano
tem_algum = df_painel.groupby("CO_MUNICIPIO")["fl_presenca_federal_ept_ativa"].any()
print(f"Municípios da Fase II (de 147) com EPT federal ativa observada em ≥1 ano: {int(tem_algum.sum())}")
print(f"Municípios da Fase II sem nenhuma presença EPT federal ativa observada  : {int((~tem_algum).sum())}")
print("\n(Observação descritiva — não é classificação de tratado/controle.)")

# 16.5 — distribuição do nº de escolas federais nos municípios-ano com presença
com_presenca = df_painel.loc[df_painel["qt_escolas_federais"] > 0, "qt_escolas_federais"]
display(com_presenca.describe().to_frame("qt_escolas_federais | município-ano com presença"))

,municipios_com_presenca_federal,municipios_com_ept,municipios_com_ept_ativa
NU_ANO_CENSO,,,
2007,6,2,2
2008,8,5,5
2009,29,27,27
2010,65,59,59
2011,132,129,129
2012,145,143,143
2013,147,146,146
2014,147,146,146
2015,147,146,146


Municípios da Fase II (de 147) com EPT federal ativa observada em ≥1 ano: 147
Municípios da Fase II sem nenhuma presença EPT federal ativa observada  : 0

(Observação descritiva — não é classificação de tratado/controle.)


,qt_escolas_federais | município-ano com presença
count,1414.0
mean,1.159123
std,0.87069
min,1.0
25%,1.0
50%,1.0
75%,1.0
max,12.0


## 17. Limitações metodológicas

- **Proxy anual de presença.** O Censo Escolar é uma foto anual. "Primeira
  presença federal EPT observada" é uma *proxy* de timing e **ainda
  precisa ser confrontada** com atos de criação, inauguração e início
  efetivo das atividades (ver `CONTRATO_CAUSAL.md`).
- **Oferta EPT ≠ atividade.** `fl_oferta_ept` (`IN_PROF == 1`) não depende
  da situação de funcionamento; para uso causal, a métrica com semântica
  de "campus operante" é `fl_presenca_federal_ept_ativa`.
- **Rótulos de nome no painel.** `NO_MUNICIPIO`/`SG_UF`/`CO_UF` vêm do
  Censo quando há escola no ano e do cadastro Fase II (IBGE) caso
  contrário; pode haver pequena diferença de grafia — a chave é sempre
  `CO_MUNICIPIO`.
- **Sem suíte de testes automatizada** (`tests/` só tem `.gitkeep`); a
  validação vem das funções `validate_*` da rotina e deste notebook.
- **`data/` não é versionada** (`.gitignore` do monorepo); a
  reprodutibilidade é garantida pelo script + manifesto com checksums.
- **Divergência 147 vs. 144** entre esta reconstrução e documentos
  congelados permanece aberta (fora do escopo desta etapa).
- **Nenhuma atribuição causal** é feita: sem coorte, sem grupo de
  controle, sem *common support*, sem ATT.

## 18. Arquivos produzidos e *hashes*

SHA-256 calculado agora sobre os Parquets em disco, comparado com os
valores registrados em
[`AUDITORIA_CONSTRUCAO_CENSO_ESCOLAR.md`](../docs/data/AUDITORIA_CONSTRUCAO_CENSO_ESCOLAR.md)
(seção 12).

In [15]:
HASHES_DOC = {
    P_ESCOLA.name: "a6ae8a3882b120ecba11b9cdef66507ded5113d46bf1d74de8a744be9630da73",
    P_MUNANO.name: "86d8e80d46da47ab6cf73d9ea3c92eb216dc61a89ce6afa67588d60aa43d94d3",
    P_PAINEL.name: "46d3b7f255e2b372116c2aee80743f1d2d5434f9fcce5e640c05dfbeb7cc8dfd",
}
def sha256(p, buf=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(buf), b""):
            h.update(b)
    return h.hexdigest()

linhas = []
for p in (P_ESCOLA, P_MUNANO, P_PAINEL):
    atual = sha256(p)
    doc = HASHES_DOC[p.name]
    linhas.append({"arquivo": p.name, "bytes": p.stat().st_size,
                   "sha256_atual": atual, "confere_com_doc": atual == doc})
display(pd.DataFrame(linhas))
print("Todos os hashes conferem com a documentação:",
      all(sha256(p) == HASHES_DOC[p.name] for p in (P_ESCOLA, P_MUNANO, P_PAINEL)))

,arquivo,bytes,sha256_atual,confere_com_doc
0,censo_escolas_federais_2007_2019.parquet,152637,a6ae8a3882b120ecba11b9cdef66507ded5113d46bf1d7...,True
1,censo_federal_municipio_ano_2007_2019.parquet,90233,86d8e80d46da47ab6cf73d9ea3c92eb216dc61a89ce6af...,True
2,painel_presenca_federal_ept_fase_ii_2007_2019....,36726,46d3b7f255e2b372116c2aee80743f1d2d5434f9fcce5e...,True


Todos os hashes conferem com a documentação: True


## 19. Conclusão da etapa

- Os três Parquets estão **íntegros** e conferem com os *hashes*
  documentados.
- Chaves únicas, sem duplicidades, sem nulos em identificadores.
- Período **2007–2019** completo nas três tabelas.
- Painel Fase II = grade completa **147 municípios × 13 anos = 1.911
  observações**, com o conjunto de municípios idêntico ao da lista Fase II.
- Flags e contagens **coerentes** com as quatro relações de contenção
  corretas; sem *mojibake* nos nomes.
- **Nenhuma atribuição causal ou de tratamento** foi feita nesta etapa.

A tabela **painel_presenca_federal_ept_fase_ii_2007_2019.parquet** está
pronta para servir de insumo às próximas etapas (revisão de literatura e
reconstrução institucional do timing), conforme o roadmap acadêmico.

## 20. (Opcional) Reprodutibilidade do pipeline completo

Este notebook usa os Parquets **já validados**. Para reconstruir tudo do
zero a partir dos ZIPs locais do Censo (leitura direta, sem download),
basta executar a rotina de produção:

```bash
python src/constroi_painel_censo_escolar.py
```

A rotina relê os 13 ZIPs, refiltra as escolas federais, reconstrói as três
tabelas, reexecuta as validações internas e regrava os Parquets (hashes
determinísticos). **Não execute agora** — a célula abaixo fica desligada
por padrão.

In [16]:
EXECUTAR_PIPELINE_COMPLETO = False  # mantenha False: o notebook usa os Parquets já validados

if EXECUTAR_PIPELINE_COMPLETO:
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "src" / "constroi_painel_censo_escolar.py")], check=True)
else:
    print("Pipeline completo NÃO reexecutado por este notebook.")
    print("Para reconstruir do zero:  python src/constroi_painel_censo_escolar.py")

Pipeline completo NÃO reexecutado por este notebook.
Para reconstruir do zero:  python src/constroi_painel_censo_escolar.py
